# 🤖 AutoML Pilot - Pro Training Template
Use this Colab notebook to run advanced AutoML (PyCaret) and generate automated reports. This template is designed to work seamlessly with datasets exported from the **AutoML Pilot** web app.

In [ ]:
# @title 1. Install Dependencies
# Install PyCaret, ydata-profiling, and other necessary libraries
!pip install -q pycaret[full] ydata-profiling pandas
print('✅ Dependencies installed!')

In [ ]:
# @title 2. Load Dataset
import pandas as pd
import os
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print(f'✅ Loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns')
display(df.head())

In [ ]:
# @title 3. Automated EDA Report
from ydata_profiling import ProfileReport

print('🔍 Generating Exploratory Data Analysis report...')
profile = ProfileReport(df, title="AutoML Pilot - Data Profile", explorative=True)
profile.to_file("data_report.html")
print('✅ EDA Report saved as data_report.html')

In [ ]:
# @title 4. AutoML Setup & Training
from pycaret.classification import setup as cls_setup, compare_models as cls_compare, pull as cls_pull, save_model as cls_save, finalize_model as cls_finalize
from pycaret.regression import setup as reg_setup, compare_models as reg_compare, pull as reg_pull, save_model as reg_save, finalize_model as reg_finalize

# Configuration
target_column = 'target' # @param {type:"string"}
task_type = 'classification' # @param ["classification", "regression"]

print(f'🚀 Starting AutoML for {task_type} on target: {target_column}')

if task_type == 'classification':
    s = cls_setup(data=df, target=target_column, session_id=123, verbose=False, html=False)
    print('Comparing models...')
    best_model = cls_compare()
    leaderboard = cls_pull()
    display(leaderboard)
    
    # Finalize and Save
    final_model = cls_finalize(best_model)
    cls_save(final_model, 'best_model')
    print('✅ Classification model saved as best_model.pkl')

else:
    s = reg_setup(data=df, target=target_column, session_id=123, verbose=False, html=False)
    print('Comparing models...')
    best_model = reg_compare()
    leaderboard = reg_pull()
    display(leaderboard)
    
    # Finalize and Save
    final_model = reg_finalize(best_model)
    reg_save(final_model, 'best_model')
    print('✅ Regression model saved as best_model.pkl')

In [ ]:
# @title 5. Email Reporting (Optional)
import smtplib
import ssl
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

SEND_EMAIL = False # @param {type:"boolean"}
RECIPIENT_EMAIL = "your-email@example.com" # @param {type:"string"}
SENDER_EMAIL = "" # @param {type:"string"}
SENDER_PASSWORD = "" # @param {type:"string"}

if SEND_EMAIL and SENDER_EMAIL and SENDER_PASSWORD:
    try:
        msg = MIMEMultipart("alternative")
        msg["Subject"] = f"AutoML Pilot - Colab Training Report ({task_type})"
        msg["From"] = SENDER_EMAIL
        msg["To"] = RECIPIENT_EMAIL
        
        metrics_html = leaderboard.to_html(classes='table table-striped')
        
        html_body = f"""
        <html>
        <body>
          <h2>🚀 AutoML Pilot - Training Results</h2>
          <p>The automated training on Colab has completed successfully.</p>
          <h3>📊 Model Leaderboard</h3>
          {metrics_html}
          <p>You can now download the best_model.pkl and import it back into the AutoML Pilot web app.</p>
        </body>
        </html>
        """
        msg.attach(MIMEText(html_body, "html"))
        
        context = ssl.create_default_context()
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=context) as server:
            server.login(SENDER_EMAIL, SENDER_PASSWORD)
            server.sendmail(SENDER_EMAIL, RECIPIENT_EMAIL, msg.as_string())
        print(f'✅ Results emailed to {RECIPIENT_EMAIL}')
    except Exception as e:
        print(f'❌ Email failed: {str(e)}')
else:
    print('ℹ️ Email reporting skipped. Configure credentials to enable.')

In [ ]:
# @title 6. Download Model & Report
from google.colab import files

if os.path.exists('best_model.pkl'):
    files.download('best_model.pkl')
    print('✅ Downloading best_model.pkl')

if os.path.exists('data_report.html'):
    files.download('data_report.html')
    print('✅ Downloading data_report.html')